# Demo — AgentCore Observability Trace Inspection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/demos/demo-observability-trace/demo-observability-trace.ipynb)

This notebook is a follow-along demo.

Day 1 — Block 4: Context and Visibility (Memory and Observability)

Demonstrates how to inspect an OpenTelemetry-compatible trace emitted by
AgentCore. Explains the difference between system-level logs and agent-level
reasoning traces.

In [ ]:
# Install required dependencies
!pip install boto3 --quiet

In [ ]:
import json

def print_section(title: str):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def show_trace_structure():
    """Analyze a structured OpenTelemetry trace from AgentCore."""
    print_section("Analyzing an AgentCore Trace")

    trace = {
        "traceId": "1-65a8b9c0-1234567890abcdef12345678",
        "name": "InvokeAgentRuntime",
        "startTime": "2026-08-02T15:00:01.000Z",
        "endTime": "2026-08-02T15:00:05.500Z",
        "attributes": {
            "agentcore.runtime.id": "arn:aws:bedrock.../TravelAgent-Prod",
            "agentcore.session.id": "session-99x",
            "agentcore.user.id": "user-finance-881",
        },
        "spans": [
            {
                "name": "Memory.Retrieve",
                "durationMs": 150,
                "status": "OK",
            },
            {
                "name": "Model.Generate",
                "durationMs": 2100,
                "attributes": {
                    "model.id": "anthropic.claude-3-sonnet",
                    "tokens.input": 850,
                    "tokens.output": 120,
                    "reasoning.stop_reason": "tool_use",
                }
            },
            {
                "name": "Gateway.InvokeTool",
                "durationMs": 650,
                "attributes": {
                    "tool.name": "get_order_status",
                    "tool.target": "arn:aws:lambda:...:function/OrderAPI",
                }
            },
            {
                "name": "Model.Generate",
                "durationMs": 1600,
                "attributes": {
                    "tokens.input": 1050,
                    "tokens.output": 85,
                    "reasoning.stop_reason": "end_turn",
                }
            }
        ]
    }

    print(f"  Trace ID: {trace['traceId']}")
    print(f"  Total Duration: 4.5 seconds")
    print("\n  Execution Timeline:")
    for span in trace['spans']:
        print(f"  → [{span['durationMs']}ms] {span['name']}")
        if "attributes" in span and "tool.name" in span["attributes"]:
            print(f"      Tool: {span['attributes']['tool.name']}")
        if "attributes" in span and "tokens.output" in span["attributes"]:
            print(f"      Tokens out: {span['attributes']['tokens.output']}")

def show_observability_routing():
    """Explain where these traces go and why it matters."""
    print_section("Where Do Traces Go?")

    routing = {
        "Destination 1": "Amazon CloudWatch (Metrics and Alarms)",
        "Destination 2": "AWS X-Ray (Visual Trace Map)",
        "Destination 3": "Amazon S3 / Kinesis (SIEM integration, e.g., Splunk/Datadog)",
    }
    for dest, desc in routing.items():
        print(f"  {dest}: {desc}")

    print("\n  Why OpenTelemetry (OTel)?")
    print("  - It's vendor-neutral. You aren't locked into CloudWatch.")
    print("  - You can pipe AgentCore traces directly into your existing APM tools.")

def highlight_pii_risk():
    """Traces often contain prompts and responses, which are high PII risks."""
    print_section("WARNING: Traces Contain Prompts (PII Risk)")

    print("  By default, deep reasoning traces include:")
    print("  - The exact text the user typed (Prompts)")
    print("  - The exact text the model generated (Completions)")
    print("  - The exact JSON arguments passed to tools")
    print()
    print("  If a user types 'My credit card is 4111...', that goes into the trace.")
    print("  Production Requirement: Disable raw prompt logging in production, OR")
    print("  ensure your log aggregation tool (e.g., Macie/DataDog) redacts PII on ingest.")

def main():
    print("Observability Trace Inspection — Instructor Demo\n")
    show_trace_structure()
    show_observability_routing()
    highlight_pii_risk()

    print_section("Key Takeaways")
    print("  1. Observability traces show the exact sequence of reasoning and tool use.")
    print("  2. OpenTelemetry allows you to use your existing APM (Datadog, Splunk, etc).")
    print("  3. Traces contain raw prompts and responses — guard against PII leaks.")

if __name__ == "__main__":
    main()
